# 26. SRA チャットボット・デモ
Notebook 25のマザーボードアーキテクチャを利用し、ipywidgetsによるチャットUIを構築します。
ユーザーの入力に基づいてリアルタイムでルーターが適切なシナプスを選択し、応答を生成します。


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import ipywidgets as widgets
from IPython.display import display, clear_output

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LLM_MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"Using device: {device}")


Using device: cpu


In [ ]:
class LLMSynapse(nn.Module):
    """汎用LLMシナプス (Neural)"""
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model)
        )
        # TinyLlamaモデルをロード
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
        self.model = AutoModelForCausalLM.from_pretrained(LLM_MODEL_ID).to(device)
        self.model.eval()
    
    def forward(self, x, text_input=None):
        if text_input is not None:
            # ユーザー入力をTinyLlamaに渡して応答を生成
            inputs = self.tokenizer.encode(text_input, return_tensors="pt").to(device)
            with torch.no_grad():
                outputs = self.model.generate(inputs, max_new_tokens=50, num_return_sequences=1)
            response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            return response
        return self.net(x)

In [3]:
class VectorDBSynapse(nn.Module):
    """事実検索シナプス (Retrieval)"""
    def __init__(self, d_model):
        super().__init__()
        self.is_neural = False
        self.db = {}
        self.d_model = d_model
        
    def add_fact(self, key, value):
        self.db[key] = value
        
    def forward(self, x, text_input=None):
        if text_input is not None:
            for k, v in self.db.items():
                if k.lower() in text_input.lower():
                    return f"DB_RESULT: {v}"
            return "DB_MISS: 知識ベースにありません"
        return x

In [4]:
class CalculatorSynapse(nn.Module):
    """ルールベース計算機シナプス"""
    def __init__(self):
        super().__init__()
        self.is_neural = False
    def forward(self, x, text_input=None):
        if text_input is not None:
            try:
                ans = eval(text_input)
                return f"CALC_RESULT: {ans}"
            except:
                return "CALC_ERROR: 有効な数式ではありません"
        return x

In [5]:
class LastTokenRouter(nn.Module):
    def __init__(self, d_model, num_synapses):
        super().__init__()
        self.classifier = nn.Linear(d_model, num_synapses, bias=False)
        
    def forward(self, x):
        last_token_embeds = x[:, -1, :]
        logits = self.classifier(last_token_embeds)
        probs = F.softmax(logits, dim=-1)
        return probs, logits

class SRAMotherboard(nn.Module):
    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.synapses = nn.ModuleDict()
        self.synapse_names = []
        self.router = None 
        
    def add_synapse(self, name, synapse_module):
        self.synapses[name] = synapse_module
        self.synapse_names.append(name)
        old_router = self.router
        self.router = LastTokenRouter(self.d_model, len(self.synapse_names)).to(device)
        if old_router is not None:
            with torch.no_grad():
                self.router.classifier.weight[:len(self.synapse_names)-1, :] = old_router.classifier.weight.data
                
    def _is_failed_output(self, output):
        return isinstance(output, str) and output.startswith(("CALC_ERROR", "DB_MISS"))

    def route_and_forward(self, input_ids, text_inputs=None):
        x = self.embedding(input_ids)
        probs, _ = self.router(x)
        
        batch_size = x.size(0)
        final_outputs = []
        routed_synapses = []
        
        for i in range(batch_size):
            sample_probs = probs[i]
            sorted_indices = torch.argsort(sample_probs, descending=True)
            text_input = text_inputs[i] if text_inputs else None
            
            for rank, syn_idx in enumerate(sorted_indices):
                selected_synapse_name = self.synapse_names[syn_idx.item()]
                synapse = self.synapses[selected_synapse_name]
                out = synapse(x[i:i+1], text_input=text_input)
                if self._is_failed_output(out):
                    continue

                prefix = f"[{selected_synapse_name}]"
                if rank > 0:
                    primary_name = self.synapse_names[sorted_indices[0].item()]
                    prefix = f"[{selected_synapse_name} fallback from {primary_name}]"
                final_outputs.append(f"{prefix} {out}")
                routed_synapses.append(selected_synapse_name)
                break
            else:
                primary_name = self.synapse_names[sorted_indices[0].item()]
                final_outputs.append(f"[{primary_name}] 全シナプスで処理に失敗しました")
                routed_synapses.append(primary_name)
                
        return final_outputs, routed_synapses[-1] if len(routed_synapses) == 1 else routed_synapses


In [6]:
vocab_size = 1000
d_model = 128

model = SRAMotherboard(d_model=d_model, vocab_size=vocab_size).to(device)

# シナプスの登録
model.add_synapse("GeneralLLM", LLMSynapse(d_model).to(device))
model.add_synapse("Calculator", CalculatorSynapse())

vdb = VectorDBSynapse(d_model)
vdb.add_fact("Japan", "Tokyo")
vdb.add_fact("CEO", "John Doe")
vdb.add_fact("SynapticRouter", "動的に最適なモジュールへルーティングする次世代AIアーキテクチャです。")
model.add_synapse("VectorDB", vdb)

# 入力テキストをダミーIDに変換する関数
def text_to_ids(text, max_len=5):
    # デモ用の簡易的な変換
    ids = [(ord(c) * 31) % vocab_size for c in text]
    if len(ids) < max_len:
        ids += [0] * (max_len - len(ids))
    return torch.tensor([ids[:max_len]]).to(device)

# デモ用にルーターを学習させる
demo_data = [
    {"text": "15 * 4", "target": 1},
    {"text": "100 / 2", "target": 1},
    {"text": "2 + 2", "target": 1},
    {"text": "Japan", "target": 2},
    {"text": "CEO", "target": 2},
    {"text": "SynapticRouter", "target": 2},
    {"text": "Hello", "target": 0},
    {"text": "こんにちは", "target": 0},
    {"text": "How are you?", "target": 0},
]

optimizer = torch.optim.Adam(model.router.parameters(), lr=0.05)
criterion = nn.CrossEntropyLoss()

print("ルーターを学習中...")
for epoch in range(100):
    optimizer.zero_grad()
    loss = 0
    for d in demo_data:
        ids = text_to_ids(d["text"])
        x = model.embedding(ids)
        _, logits = model.router(x)
        loss += criterion(logits, torch.tensor([d["target"]]).to(device))
    loss.backward()
    optimizer.step()
print("学習完了！")


ルーターを学習中...
学習完了！


## チャットボット UI
以下のセルを実行すると、チャット画面が表示されます。
数式（例: `15 * 4`）や、知識（例: `Japan`, `SynapticRouter`）、または一般的な挨拶（例: `Hello`）を入力してみてください。


In [7]:
# チャットUIの構築
chat_output = widgets.Output()

text_input = widgets.Text(
    value='',
    placeholder='メッセージを入力... (例: 15 * 4, Japan, Hello)',
    description='You:',
    disabled=False,
    layout=widgets.Layout(width='80%')
)

send_button = widgets.Button(
    description='送信',
    disabled=False,
    button_style='info', 
    tooltip='Click to send',
    icon='paper-plane'
)

def on_send_clicked(b):
    user_text = text_input.value
    if not user_text:
        return
        
    with chat_output:
        print(f"👤 あなた: {user_text}")
        
        # 入力をIDに変換してルーティング・推論
        ids = text_to_ids(user_text)
        outputs, routed_synapse = model.route_and_forward(ids, text_inputs=[user_text])
        
        # 応答の表示
        print(f"🤖 SRA (via {routed_synapse}): {outputs[0].split('] ')[1]}")
        print("-" * 50)
        
    text_input.value = ''

send_button.on_click(on_send_clicked)
text_input.on_submit(on_send_clicked)

display(widgets.VBox([chat_output, widgets.HBox([text_input, send_button])]))


/var/folders/49/rfxswdk900v3bb93_pqsnjyw0000gn/T/ipykernel_56412/1444747971.py:39: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  text_input.on_submit(on_send_clicked)
